In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import ProcessingCode as pc

### Process model data to make plotting notebooks smaller 

In [2]:
ds = xr.open_mfdataset("SPEC/*2023*.nc")

In [3]:
ds

<xarray.Dataset> Size: 1GB
Dimensions:       (time: 7553, station: 28, string16: 16, frequency: 36,
                   direction: 36)
Coordinates:
  * time          (time) datetime64[ns] 60kB 2023-01-04 ... 2023-09-30T18:00:00
  * station       (station) float64 224B 1.0 2.0 3.0 4.0 ... 25.0 26.0 27.0 28.0
  * string16      (string16) float64 128B nan nan nan nan ... nan nan nan nan
  * frequency     (frequency) float32 144B 0.035 0.0385 ... 0.8942 0.9836
  * direction     (direction) float32 144B 89.13 79.13 69.13 ... 109.1 99.13
Data variables:
    station_name  (time, station, string16) |S1 3MB dask.array<chunksize=(7, 28, 16), meta=np.ndarray>
    longitude     (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    latitude      (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    frequency1    (time, frequency) float32 1MB dask.array<chunksize=(7, 36), meta=np.ndarray>
    frequency2    (time, frequency) float32 1MB dask.array<chunksize=(7, 36), meta=np.ndarray>
    efth          (time, station, frequency, direction) float32 1GB dask.array<chunksize=(1, 28, 36, 36), meta=np.ndarray>
    dpt           (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    wnd           (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    wnddir        (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    cur           (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
    curdir        (time, station) float32 846kB dask.array<chunksize=(1, 28), meta=np.ndarray>
Attributes: (12/16)
    product_name:           ww3.2023_spec.nc
    area:                   West Coast
    data_type:              OCO spectra 2D
    format_version:         1.1
    southernmost_latitude:  n/a
    northernmost_latitude:  n/a
    ...                     ...
    minimum_altitude:       n/a
    maximum_altitude:       n/a
    altitude_resolution:    n/a
    start_date:             2023-01-04 00:00:00
    stop_date:              2023-01-04 06:00:00
    field_type:             hourly

In [4]:
xr_stn = ds['station_name']
stn_str = xr_stn[0].to_numpy()
str_stns = []

for i in range(len(stn_str[:])):
    str_stns.append(pc.toNameString(stn_str[i][:]))

In [5]:
for i in range(len(str_stns)):
    print(i,str_stns[i])

0 C46004-Mid
1 C46131-Sen
2 C46145-Cen
3 C46146-Hal
4 C46147-Sou
5 C46184-Nor
6 C46185-Sou
7 C46204-Wes
8 C46205-Wes
9 C46206-La
10 C46207-Eas
11 C46208-Wes
12 Near-Shore
13 Amphitrite
14 46087-NOAA
15 46088-NOAA
16 MarineLabs
17 C46036-Sou
18 Spotter
19 Trans-pnt0
20 Trans-pnt1
21 Trans-pnt2
22 Trans-pnt3
23 Trans-pnt4
24 Trans-pnt5
25 Trans-pnt6
26 Trans-pnt7
27 Trans-pnt8


### Separate the stations into their categories: Open Ocean, Open Coastal,  Sheltered and Strait of Georgia

In [6]:
mod_time = ds["time"].values
mod_efth = ds["efth"].values
mod_fq = ds["frequency"].values

In [7]:
mod_dir = ds["direction"].values

In [8]:
Sf_mod = []
for s in range(len(str_stns)):
    Sf_mod.append(pc.passiton(mod_efth,s))

In [9]:
np.shape(Sf_mod)

(28, 7553, 36)

In [10]:
def spectralMoment(Sf,f,tag,n,t):
  
    m_n = []
    for i in range(len(t)):
        if tag=='meds' or tag=='noaa' or tag=='amph':
            start = 0
            stop = 27
            m_n.append(np.trapz(Sf[i][start:stop]*f[start:stop]**n,f[start:stop]))
        elif tag=='spot':
            start = 0
            stop = None
            m_n.append(np.trapz(Sf[i][start:stop]*f[start:stop]**n,f[start:stop]))
        elif tag=='r':
            start = 5
            stop = None
            m_n.append(np.trapz(Sf[i][start:stop]*f[start:stop]**n,f[start:stop]))
    return m_n

In [11]:
def T01(t, m0, m1):
        
    t01 = []
    f01 = []
    for i in range(len(t)):
        t01.append(m0[i]/m1[i])
        f01.append(1/t01[i])
        
    return t01,f01

In [12]:
def Hs(t, m0):
    
    hs = []
    for i in range(len(t)):
        hs.append(4*np.sqrt(m0[i]))
        
    return hs

In [13]:
def Bandwidth(t, m0, m1, m2):
    
    sig = []
    for i in range(len(t)):
        sig.append(np.sqrt((m0[i]*m2[i]/m1[i]**2)-1))
        
    return sig

In [14]:
def Saturation(t, m4):
    
    sat = []
    for i in range(len(t)):
        sat.append(((2*np.pi)**4*m4[i])*(1/(2*9.8**2)))
        
    return sat

In [15]:
def CTCC(t, Sf,f,T01):

    tau = []
    rho,lmda = [], []
    r = []
    fs = 5
    fe = None
    m1 = spectralMoment(Sf,f,'r',1,t)
    m0= spectralMoment(Sf,f,'r',0,t)
    
    for i in range(len(t)):
        
        if np.isnan(T01[i]):
            T01[i]=0
            
        tau.append(T01[i]/2)
        
        rho.append(np.trapz(Sf[i][fs:fe]*np.cos(2*np.pi*np.asarray(f[fs:fe])*np.asarray(tau[i])),f[fs:fe]))
        lmda.append(np.trapz(Sf[i][fs:fe]*np.sin(2*np.pi*np.asarray(f[fs:fe])*np.asarray(tau[i])),f[fs:fe]))
        
        r.append((1/m0[i])*np.sqrt(rho[i]**2+lmda[i]**2))
        if np.isnan(r[i]):
            r[i]=0
         
    return r

In [16]:
def find_nearest(fq_ar, f01_val):
    array = np.asarray(fq_ar)
    idx = (np.abs(array - f01_val)).argmin()
    return idx

In [17]:
def Spreading(time, efth, theta, fq, Sf):
    
    theta_proper = (180+theta)%360
    #sort_idx = np.argsort(theta_proper)
    #theta_sorted = theta_proper[sort_idx]
    theta_rad = np.deg2rad(theta_proper)
    
    
    #efth_sorted = efth[:, :, sort_idx]

    a1 = []
    b1 = []
    E = []
    
    for i in range(len(time)):
        a1.append(np.trapz(np.trapz(efth[i]*np.cos(theta_rad),fq, axis=0),dx=2*np.pi/36))
        
        b1.append(np.trapz(np.trapz(efth[i]*np.sin(theta_rad),fq, axis=0),dx=2*np.pi/36))
        
        E.append(np.trapz(Sf[i],fq))
    a = np.array(a1)
    b = np.array(b1)
    r = np.sqrt(a**2 + b**2)
    
    mean_spr = np.sqrt(2*(1-(r/np.sqrt(np.array(E)**2))))
    wv_dir = np.rad2deg(np.arctan2(b,a))
    
    #wv_dir[wv_dir<0]+=360
    mean_wv_dir = (wv_dir + 360)%360
    mean_spread = np.rad2deg(mean_spr)
    
    return mean_wv_dir, mean_spread

In [18]:
np.shape(Sf_mod)

(28, 7553, 36)

In [19]:
m0_mod = []
m1_mod = []
m2_mod = []
m3_mod = []
m4_mod = []

for i in range(len(str_stns)):
    if i<=11 or i==17:
        tag='meds'
        m0_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,0,mod_time))
        m1_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,1,mod_time))
        m2_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,2,mod_time))
        m3_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,3,mod_time))
        m4_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,4,mod_time))
    elif i==12 or i==13:
        tag='amph'
        m0_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,0,mod_time))
        m1_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,1,mod_time))
        m2_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,2,mod_time))
        m3_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,3,mod_time))
        m4_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,4,mod_time))
    elif i==14 or i==15:
        tag='noaa'
        m0_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,0,mod_time))
        m1_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,1,mod_time))
        m2_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,2,mod_time))
        m3_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,3,mod_time))
        m4_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,4,mod_time))
    else:
        tag='spot'
        m0_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,0,mod_time))
        m1_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,1,mod_time))
        m2_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,2,mod_time))
        m3_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,3,mod_time))
        m4_mod.append(spectralMoment(Sf_mod[i],mod_fq,tag,4,mod_time))

In [20]:
hs_mod = []

for i in range(len(str_stns)):
    hs_mod.append(Hs(mod_time, m0_mod[i]))

In [21]:
t01_mod = []
f01_mod = []

for i in range(len(str_stns)):
    t01_mod.append(T01(mod_time, m0_mod[i], m1_mod[i])[0])
    f01_mod.append(T01(mod_time, m0_mod[i], m1_mod[i])[1])

In [22]:
sig_mod = []

for i in range(len(str_stns)):
    sig_mod.append(Bandwidth(mod_time, m0_mod[i], m1_mod[i], m2_mod[i]))

In [23]:
sat_mod = []

for i in range(len(str_stns)):
    sat_mod.append(Saturation(mod_time, m4_mod[i]))

In [24]:
r_mod = []

for i in range(len(str_stns)):
    r_mod.append(CTCC(mod_time, Sf_mod[i], mod_fq, t01_mod[i]))

In [25]:
spr_mean_mod = []
mean_wave = []
for i in range(len(str_stns)):
    spr_mean_mod.append(Spreading(mod_time, mod_efth[:,i], mod_dir, mod_fq,Sf_mod[i])[1])
    mean_wave.append(Spreading(mod_time, mod_efth[:,i], mod_dir, mod_fq,Sf_mod[i])[0])

In [26]:
mod_efth = np.transpose(mod_efth,(1,0,2,3))

In [34]:
mod_efth = np.array(mod_efth)[:19,:,:,:]
Sf_mod = np.array(Sf_mod)[:19,:,:]
m0_mod = np.array(m0_mod)[:19,:]
m1_mod = np.array(m1_mod)[:19,:]
m2_mod = np.array(m2_mod)[:19,:]
m3_mod = np.array(m3_mod)[:19,:]
m4_mod = np.array(m4_mod)[:19,:]
hs_mod = np.array(hs_mod)[:19,:]
t01_mod = np.array(t01_mod)[:19,:]
sig_mod = np.array(sig_mod)[:19,:]
sat_mod = np.array(sat_mod)[:19,:]
r_mod = np.array(r_mod)[:19,:]
spr_mean_mod = np.array(spr_mean_mod)[:19,:]
mean_wave = np.array(mean_wave)[:19,:]
str_stns = str_stns[:19]

In [35]:
str_stns

['C46004-Mid',
 'C46131-Sen',
 'C46145-Cen',
 'C46146-Hal',
 'C46147-Sou',
 'C46184-Nor',
 'C46185-Sou',
 'C46204-Wes',
 'C46205-Wes',
 'C46206-La',
 'C46207-Eas',
 'C46208-Wes',
 'Near-Shore',
 'Amphitrite',
 '46087-NOAA',
 '46088-NOAA',
 'MarineLabs',
 'C46036-Sou',
 'Spotter']

In [36]:
ns = xr.Dataset(
    data_vars={
        "efth": (("station","time", "frequency", "direction"), mod_efth),
        "ef": (("station", "time", "frequency"), Sf_mod),
        "m0": (("station","time"), m0_mod),
        "m1": (("station","time"), m1_mod),
        "m2": (("station","time"), m2_mod),
        "m3": (("station","time"), m3_mod),
        "m4": (("station","time"), m4_mod),
        "Hs": (("station","time"), hs_mod),
        "T01": (("station","time"), t01_mod),
        "Narrowness": (("station","time"), sig_mod),
        "Saturation": (("station","time"), sat_mod),
        "r": (("station","time"), r_mod),
        "mean_spread": (("station","time"), spr_mean_mod),
        "mean_wv_dir": (("station","time"), mean_wave),
    },
    
    coords={"time": mod_time,
            "station": str_stns,
            "frequency": mod_fq, 
            "direction": mod_dir
           },
)

# Add attribute to indicate what "bounds" means
ns["mean_spread"].attrs={"description":"Directional spreading around the mean spectral frequency at the 0 and 1 moments"}
ns["mean_wv_dir"].attrs={"description":"Mean wave direction over all frequencies. Degrees from North"}


# Save to a NetCDF file
ns.to_netcdf("WW3_1km_CUR_WLV_processed.nc")

In [ ]:
#Open ocean stations by index are: 0, 5, and 17

In [ ]:
#Open Coastal stations are: 4, 7, 8, 9, 10, 11, 13, and 18

In [ ]:
#Sheltered by index are: 2, 6, and 14

In [ ]:
#Strait of Georgia are: 1, 3, and 15